# V-01. 세계의 화산대

「지구과학 데이터 사이언스」 책의 코드입니다. 설명은 책을 참고하세요.

**준비** (책에는 싣지 않은 공통 셀)

In [ ]:
# 필요한 모듈 정의 (버전 명시 가능)
packages_str = "numpy, pandas, matplotlib, basemap, cartopy, folium, plotly, lxml, nbformat, version_information"

In [ ]:
# 공통 함수(dses_utils) 불러오기 — 코랩/로컬 모두 동작(별도 설치 불필요)
import os, urllib.request
try:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Kiehyun/dses_utils/main/dses_utils.py", "dses_utils.py")
except Exception:
    pass  # 인터넷이 안 되면 기존에 받아둔 dses_utils.py 사용
import dses_utils as du
du.check_and_install(packages_str)
# 한글 글꼴 설정 — 공통 함수(dses_utils.set_korean_font)로 통일
# (반복되던 OS별 rcParams 설정 코드는 dses_utils.py 로 옮겼습니다)
import dses_utils as du
du.set_korean_font()
# 프로젝트(데이터 저장) 폴더 생성 — 공통 함수 사용
proj_dir = du.make_project_dir(du.current_notebook_name())
from pathlib import Path

**code 1.**

In [ ]:
from pathlib import Path

# 세계 화산 목록(VAAC용) 엑셀 파일, 구글 드라이브 파일 ID
fname = "List_of_Volcanoes_of_the_World_for_VAAC_Use.xls"
fid = "17wbUIiJCxRz3XIgz6ZUM_1o4D2jOSA5P"

du.download(f"https://docs.google.com/uc?export=download&id={fid}", Path(proj_dir) / fname)

**code 2.**

In [ ]:
fpaths = du.list_files(proj_dir, "xls")

**code 3.**

In [ ]:
du.preview_text(fpaths[0], n=10);  # 앞 10줄(인코딩 자동)

**code 4.**

In [ ]:
import pandas as pd
import lxml.etree as ET

# 1) 엑셀 2003 XML(SpreadsheetML) 파일 파싱
tree = ET.parse(fpaths[0])
ns = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}  # 엑셀 XML 이름공간

# 2) 행(Row)마다 셀 값 추출
rows = tree.xpath("//ss:Row", namespaces=ns)

data = []
for row in rows:
    row_data = [cell.text for cell in row.xpath(".//ss:Cell/ss:Data", namespaces=ns)]
    data.append(row_data)

# 3) DataFrame으로 변환
df_vol = pd.DataFrame(data)

# 둘째 행을 열 이름으로, 앞 두 행(메타 정보 · 머리글)은 제거
df_vol.columns = df_vol.iloc[1]
df_vol = df_vol.drop([0, 1]).reset_index(drop=True)

# 4) 파일 끝의 메타 정보(설명) 행은 좌표가 없으므로 제외
lat = pd.to_numeric(df_vol['LatitudeDecimal'], errors='coerce')
print(f"전체 행: {len(df_vol)}, 좌표가 없는 행: {lat.isna().sum()}")
df_vol = df_vol[lat.notna()].reset_index(drop=True)

# 5) 모든 값이 숫자인 열은 숫자형으로
for col in df_vol.columns:
    num = pd.to_numeric(df_vol[col], errors='coerce')
    if num.notna().sum() == df_vol[col].notna().sum():
        df_vol[col] = num

print(f"화산 수: {len(df_vol)}")
print("df_vol :", df_vol)

**code 5.**

In [ ]:
df_vol.columns

**code 6.**

In [ ]:
# 십진 도(°) 단위 위도 · 경도를 짧은 열 이름으로
df_vol['Latitude'] = df_vol['LatitudeDecimal']
df_vol['Longitude'] = df_vol['LongitudeDecimal']
print(df_vol[['Latitude', 'Longitude']])

**code 7.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 화산 위치 산점도
plt.scatter(df_vol['Longitude'], df_vol['Latitude'], 
            c='red', s=15, alpha=0.6, marker='^', 
            edgecolors='darkred', linewidth=0.3)

# 축 범위 · 기준선
plt.xlim(-180, 180)  # 경도(°)
plt.ylim(-90, 90)    # 위도(°)
plt.xlabel('경도', fontsize=12)
plt.ylabel('위도', fontsize=12)
plt.title('전 세계 화산 분포', fontsize=16, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)  # 적도(위도 0°)
plt.axvline(x=0, color='k', linestyle='-', alpha=0.3)  # 본초자오선(경도 0°)

plt.tight_layout()
du.save_figure(plt.gcf(), os.path.join(proj_dir, "05_01_Plotting_ring_of_fire_01"))
plt.show()

**code 8.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
import numpy as np

print(f"총 화산 개수: {len(df_vol)}")

plt.figure(figsize=(12.0, 8.0))

# 원통 도법(cyl) 세계 지도
m = Basemap(projection='cyl', resolution='c',
            llcrnrlat=-90, urcrnrlat=90,    # 왼쪽 아래 · 오른쪽 위 모서리의 위도
            llcrnrlon=-180, urcrnrlon=180)  # 같은 모서리의 경도

# 지도 기본 요소 그리기
m.drawcoastlines(linewidth=0.5)
m.drawcountries(linewidth=0.5)
m.drawmapboundary(fill_color='lightblue')  # 바다 색
m.fillcontinents(color='lightgray', lake_color='lightblue')

m.drawparallels(np.arange(-90, 91, 30), labels=[1,0,0,0], fontsize=8)  # 30° 간격, 왼쪽 눈금
m.drawmeridians(np.arange(-180, 181, 60), labels=[0,0,0,1], fontsize=8)  # 60° 간격, 아래 눈금

# 경위도 → 지도 좌표
x, y = m(df_vol['Longitude'].values, df_vol['Latitude'].values)

m.scatter(x, y, c='red', s=20, alpha=0.7, marker='^', 
            edgecolors='darkred', linewidth=0.5, label='화산')

plt.title('전 세계 화산 분포도', fontsize=16, fontweight='bold')
plt.legend(loc='lower left')
plt.tight_layout()
du.save_figure(plt.gcf(), os.path.join(proj_dir, "05_01_Plotting_ring_of_fire_02"))
plt.show()

**code 9.**

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

plt.figure(figsize=(12.0, 6.4))
ax = plt.axes(projection=ccrs.PlateCarree())  # 정거 원통 도법
ax.set_global()                               # 지구 전체
ax.coastlines()
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax.add_feature(cfeature.BORDERS, linestyle=':')
plt.scatter(df_vol['Longitude'], df_vol['Latitude'], 
            c='red', s=15, alpha=0.6, marker='^', 
            edgecolors='darkred', linewidth=0.3, 
            transform=ccrs.PlateCarree(), label='화산')  # 데이터 좌표: 경위도
plt.title('전 세계 화산 분포 (Cartopy)', fontsize=16, fontweight='bold')
plt.legend(loc='lower left')
du.save_figure(plt.gcf(), os.path.join(proj_dir, "05_01_Plotting_ring_of_fire_03"))
plt.show()

**code 10.**

In [ ]:
import folium
from folium.plugins import MarkerCluster
import pandas as pd

# 지도 중심: 서울 부근(위도 37.5°, 경도 127°)
m = folium.Map(location=[37.5, 127], zoom_start=2, tiles='cartodbpositron')
marker_cluster = MarkerCluster().add_to(m)  # 가까운 마커끼리 묶어 표시
for idx, row in df_vol.iterrows():
    lat = row['Latitude']
    lon = row['Longitude']
    name = row['VolcanoName'] if 'VolcanoName' in df_vol.columns else ''
    # 위경도가 없는 화산은 건너뜀
    if pd.isna(lat) or pd.isna(lon):
        continue
    folium.Marker([lat, lon], popup=name, icon=folium.Icon(color='red', icon='fire')).add_to(marker_cluster)
m.save('world_volcanoes_folium.html')
display(m)

**code 11.**

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

fig = go.Figure()

# 화산 위치 추가
fig.add_trace(go.Scattergeo(
            lon=df_vol['Longitude'],
            lat=df_vol['Latitude'],
            mode='markers',
            marker=dict(
                size=8,
                color='red',
                symbol='triangle-up',
                line=dict(width=1, color='darkred')
            ),
            # 화산 이름(없으면 생략)
            text=df_vol['VolcanoName'] if 'VolcanoName' in df_vol.columns else None,
            # 마우스를 올렸을 때 보일 정보
            hovertemplate='<b>%{text}</b><br>' +
                            '위도: %{lat}<br>' +
                            '경도: %{lon}<br>' +
                            '<extra></extra>',
            name='화산'
        ))

# 지도 레이아웃 설정
fig.update_layout(
        title='전 세계 화산 분포 (Plotly)',
        geo=dict(
            showland=True,
            landcolor='rgb(243, 243, 243)',
            coastlinecolor='rgb(204, 204, 204)',
            showocean=True,
            oceancolor='rgb(230, 245, 255)',
            projection_type='natural earth'  # 지도 도법
        ),
        width=800,  # 그림 크기(픽셀)
        height=450
    )

fig.show()